# 3B.3 Inter-Rater Agreement - Cohen's Kappa

## Objective
Measure how well the LLM Judge aligns with **human judgment** using Cohen's Kappa.

## Methodology

1. Manually label a subset of ~10 event pairs as "Consistent" or "Contradictory"
2. Run the LLM Judge on the same pairs
3. Convert judge scores to binary (e.g., ≥70 = Consistent, <70 = Contradictory)
4. Calculate **Cohen's Kappa** between human and LLM labels


In [15]:
# Setup and Imports
import os
import json
import time
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.metrics import cohen_kappa_score, confusion_matrix, classification_report
from dotenv import load_dotenv

# Load API key from root .env file
load_dotenv(Path("../.env"))

from google import genai
from google.genai import types

client = genai.Client(api_key=os.getenv('GEMINI_API_KEY'))
MODEL = "gemini-2.5-flash-lite"

print(f"✓ Using model: {MODEL}")


✓ Using model: gemini-2.5-flash-lite


In [16]:
# Load Prompt Templates
PROMPT_DIR = Path(".")

def load_prompt(filename: str) -> str:
    filepath = PROMPT_DIR / filename
    with open(filepath, 'r', encoding='utf-8') as f:
        return f.read().strip()

ZERO_SHOT_PROMPT = load_prompt("Zero_Shot_Prompt.md")
COT_PROMPT = load_prompt("CoT_Prompt.md")
FEW_SHOT_PROMPT = load_prompt("Few_Shot_Prompt.md")

PROMPTS = {
    'zero_shot': ZERO_SHOT_PROMPT,
    'chain_of_thought': COT_PROMPT,
    'few_shot': FEW_SHOT_PROMPT
}

print(f"✓ Loaded {len(PROMPTS)} prompt strategies: {list(PROMPTS.keys())}")

✓ Loaded 3 prompt strategies: ['zero_shot', 'chain_of_thought', 'few_shot']


In [17]:
# LLM Judge Function
def call_judge(pair: Dict, prompt_type: str = 'zero_shot', temperature: float = 0.0) -> Dict:
    """
    Call the LLM judge with specified prompt type.
    
    Returns:
        {
            "consistency_score": int,
            "contradictions": [...],
            "reasoning": str,
            "error": str or None
        }
    """
    prompt_template = PROMPTS[prompt_type]
    
    lincoln_claims_str = "\n".join(f"- {c}" for c in pair['lincoln_claims'])
    author_claims_str = "\n".join(f"- {c}" for c in pair['author_claims'])
    
    prompt = prompt_template.format(
        event_name=pair['event'],
        lincoln_claims=lincoln_claims_str,
        author_claims=author_claims_str,
        author_name=pair['author_name']
    )
    
    try:
        response = client.models.generate_content(
            model=MODEL,
            contents=prompt,
            config=types.GenerateContentConfig(
                temperature=temperature,
                response_mime_type="application/json"
            )
        )
        
        raw_text = response.text.strip()
        result = json.loads(raw_text)
        result['error'] = None
        return result
        
    except Exception as e:
        return {
            "consistency_score": None,
            "contradictions": [],
            "reasoning": None,
            "error": str(e)
        }

print("✓ Judge function defined")

✓ Judge function defined


## Load Human Labels and Match with Comparison Pairs

Load the manually annotated labels from `manual_labels.json` and find the corresponding comparison pairs.

In [19]:
# Load Human Labels
with open("manual_labels.json", 'r', encoding='utf-8') as f:
    manual_labels = json.load(f)

print(f"✓ Loaded {len(manual_labels['annotations'])} human annotations")
print(f"  Rating scale: {manual_labels['metadata']['rating_scale']}")

# Load comparison pairs
with open("comparisons.json", 'r', encoding='utf-8') as f:
    all_comparisons = json.load(f)

print(f"✓ Loaded {len(all_comparisons)} total comparison pairs")

# Match human labels with comparison pairs
labeled_pairs = []

for annotation in manual_labels['annotations']:
    event = annotation['event']
    author = annotation['author']
    
    # Find matching pair in comparisons
    matching_pair = None
    for pair in all_comparisons:
        if pair['event'] == event and pair['author_name'] == author:
            matching_pair = pair
            break
    
    if matching_pair:
        labeled_pairs.append({
            'id': annotation['id'],
            'event': event,
            'author': author,
            'human_rating': annotation['rating'],
            'human_notes': annotation['notes'],
            'lincoln_claims': matching_pair['lincoln_claims'],
            'author_claims': matching_pair['author_claims'],
            'author_name': matching_pair['author_name']
        })
    else:
        print(f"⚠ No match found for: {event} - {author}")

print(f"\n✓ Matched {len(labeled_pairs)} pairs for comparison")

# Display matched pairs
print("\n" + "=" * 60)
print("PAIRS FOR INTER-RATER AGREEMENT:")
print("=" * 60)
for p in labeled_pairs:
    print(f"[{p['id']}] {p['event']} - {p['author'][:30]}...")
    print(f"    Human Rating: {p['human_rating']}")

✓ Loaded 10 human annotations
  Rating scale: ['Consistent', 'Not Consistent']
✓ Loaded 20 total comparison pairs

✓ Matched 10 pairs for comparison

PAIRS FOR INTER-RATER AGREEMENT:
[1] Election Night 1860 - John T. Morse Jr....
    Human Rating: Consistent
[2] Election Night 1860 - Baron Godfrey Rathbone Benson ...
    Human Rating: Consistent
[3] Fort Sumter Decision - Francis F. Browne...
    Human Rating: Consistent
[4] Fort Sumter Decision - Henry Ketcham...
    Human Rating: Consistent
[5] Gettysburg Address - John Hay, John G. Nicolay...
    Human Rating: Not Consistent
[6] Gettysburg Address - Baron Godfrey Rathbone Benson ...
    Human Rating: Consistent
[7] Second Inaugural Address - John T. Morse Jr....
    Human Rating: Consistent
[8] Second Inaugural Address - John Hay, John G. Nicolay...
    Human Rating: Not Consistent
[9] Gettysburg Address - Francis F. Browne...
    Human Rating: Consistent
[10] Second Inaugural Address - Baron Godfrey Rathbone Benson ...
    Human Ra

## Run LLM Judge on Labeled Pairs (All Strategies)

Run the LLM judge using **all three prompt strategies** on the same pairs that were manually labeled.

In [20]:
# Run LLM Judge on Labeled Pairs - ALL STRATEGIES
DELAY_SECONDS = 4
PROMPT_TYPES = ['zero_shot', 'chain_of_thought', 'few_shot']

print(f"Running LLM Judge on {len(labeled_pairs)} pairs...")
print(f"Prompt strategies: {PROMPT_TYPES}")
print(f"Temperature: 0.0 (deterministic)")
print(f"Total API calls: {len(labeled_pairs) * len(PROMPT_TYPES)}")
print("-" * 60)

results = []

for i, pair in enumerate(labeled_pairs):
    print(f"\n[{i+1}/{len(labeled_pairs)}] {pair['event']} - {pair['author'][:25]}...")
    
    pair_result = {
        'id': pair['id'],
        'event': pair['event'],
        'author': pair['author'],
        'human_rating': pair['human_rating'],
    }
    
    for prompt_type in PROMPT_TYPES:
        # Call the judge
        judgment = call_judge(pair, prompt_type=prompt_type, temperature=0.0)
        
        if judgment['error']:
            print(f"    {prompt_type}: ⚠ Error")
            pair_result[f'{prompt_type}_score'] = None
        else:
            score = judgment.get('consistency_score')
            pair_result[f'{prompt_type}_score'] = score
            print(f"    {prompt_type}: {score}")
        
        time.sleep(DELAY_SECONDS)
    
    print(f"    Human: {pair['human_rating']}")
    results.append(pair_result)

print("\n" + "=" * 60)
print("✓ LLM Judge complete for all strategies!")

# Create DataFrame
df = pd.DataFrame(results)
print(f"\nResults shape: {df.shape}")
print(df[['id', 'event', 'human_rating', 'zero_shot_score', 'chain_of_thought_score', 'few_shot_score']].to_string())

Running LLM Judge on 10 pairs...
Prompt strategies: ['zero_shot', 'chain_of_thought', 'few_shot']
Temperature: 0.0 (deterministic)
Total API calls: 30
------------------------------------------------------------

[1/10] Election Night 1860 - John T. Morse Jr....
    zero_shot: 75
    chain_of_thought: 75
    few_shot: 40
    Human: Consistent

[2/10] Election Night 1860 - Baron Godfrey Rathbone Be...
    zero_shot: 75
    chain_of_thought: 75
    few_shot: 75
    Human: Consistent

[3/10] Fort Sumter Decision - Francis F. Browne...
    zero_shot: 75
    chain_of_thought: 75
    few_shot: 75
    Human: Consistent

[4/10] Fort Sumter Decision - Henry Ketcham...
    zero_shot: 75
    chain_of_thought: 75
    few_shot: 75
    Human: Consistent

[5/10] Gettysburg Address - John Hay, John G. Nicolay...
    zero_shot: 40
    chain_of_thought: 75
    few_shot: 40
    Human: Not Consistent

[6/10] Gettysburg Address - Baron Godfrey Rathbone Be...
    zero_shot: 95
    chain_of_thought: 95
    f

## Calculate Cohen's Kappa

Convert LLM scores to binary labels and calculate inter-rater agreement.

In [21]:
# Convert to Binary Labels for Cohen's Kappa - ALL STRATEGIES
THRESHOLD = 70  # LLM score >= 70 = Consistent, < 70 = Not Consistent

# Convert human ratings to binary (1 = Consistent, 0 = Not Consistent)
df['human_binary'] = df['human_rating'].apply(lambda x: 1 if x == 'Consistent' else 0)

# Convert LLM scores to binary for each strategy
for strategy in ['zero_shot', 'chain_of_thought', 'few_shot']:
    col = f'{strategy}_score'
    df[f'{strategy}_binary'] = df[col].apply(lambda x: 1 if x is not None and x >= THRESHOLD else 0)

print(f"Total comparisons: {len(df)}")
print(f"Threshold for LLM: {THRESHOLD} (score >= {THRESHOLD} = Consistent)")

# Display comparison table for all strategies
print("\n" + "=" * 100)
print("HUMAN vs LLM COMPARISON (ALL STRATEGIES)")
print("=" * 100)
print(f"{'ID':<4} {'Event':<20} {'Human':<12} {'ZS':<6} {'CoT':<6} {'FS':<6} {'ZS Match':<10} {'CoT Match':<10} {'FS Match'}")
print("-" * 100)

for _, row in df.iterrows():
    human = "Con" if row['human_binary'] == 1 else "NotCon"
    zs = row['zero_shot_score'] if row['zero_shot_score'] else '-'
    cot = row['chain_of_thought_score'] if row['chain_of_thought_score'] else '-'
    fs = row['few_shot_score'] if row['few_shot_score'] else '-'
    
    zs_match = "✓" if row['human_binary'] == row['zero_shot_binary'] else "✗"
    cot_match = "✓" if row['human_binary'] == row['chain_of_thought_binary'] else "✗"
    fs_match = "✓" if row['human_binary'] == row['few_shot_binary'] else "✗"
    
    print(f"{row['id']:<4} {row['event'][:19]:<20} {human:<12} {str(zs):<6} {str(cot):<6} {str(fs):<6} {zs_match:<10} {cot_match:<10} {fs_match}")

Total comparisons: 10
Threshold for LLM: 70 (score >= 70 = Consistent)

HUMAN vs LLM COMPARISON (ALL STRATEGIES)
ID   Event                Human        ZS     CoT    FS     ZS Match   CoT Match  FS Match
----------------------------------------------------------------------------------------------------
1    Election Night 1860  Con          75     75     40     ✓          ✓          ✗
2    Election Night 1860  Con          75     75     75     ✓          ✓          ✓
3    Fort Sumter Decisio  Con          75     75     75     ✓          ✓          ✓
4    Fort Sumter Decisio  Con          75     75     75     ✓          ✓          ✓
5    Gettysburg Address   NotCon       40     75     40     ✓          ✗          ✓
6    Gettysburg Address   Con          95     95     95     ✓          ✓          ✓
7    Second Inaugural Ad  Con          95     95     85     ✓          ✓          ✓
8    Second Inaugural Ad  NotCon       75     80     55     ✗          ✗          ✓
9    Gettysburg Address

In [ ]:
# Calculate Cohen's Kappa and Other Metrics - ALL STRATEGIES
human_labels = df['human_binary'].values

# Calculate metrics for each strategy
results_summary = []
strategy_kappas = {}

for strategy in ['zero_shot', 'chain_of_thought', 'few_shot']:
    llm_labels = df[f'{strategy}_binary'].values
    
    kappa = cohen_kappa_score(human_labels, llm_labels)
    agreement = np.mean(human_labels == llm_labels)
    cm = confusion_matrix(human_labels, llm_labels)
    
    strategy_kappas[strategy] = kappa
    
    results_summary.append({
        'Strategy': strategy,
        'Kappa': kappa,
        'Agreement': agreement,
        'TN': cm[0,0], 'FP': cm[0,1], 'FN': cm[1,0], 'TP': cm[1,1]
    })

summary_df = pd.DataFrame(results_summary)

print("=" * 70)
print("INTER-RATER AGREEMENT RESULTS (ALL STRATEGIES)")
print("=" * 70)

print("\n📊 Cohen's Kappa by Strategy:")
print("-" * 50)
for _, row in summary_df.iterrows():
    print(f"   {row['Strategy']:<20}: κ = {row['Kappa']:.3f} | Agreement: {row['Agreement']:.1%}")

# Find best strategy
best_strategy = summary_df.loc[summary_df['Kappa'].idxmax()]
print(f"\n🏆 BEST STRATEGY: {best_strategy['Strategy']} (κ = {best_strategy['Kappa']:.3f})")

# Confusion matrices
print("\n" + "=" * 70)
print("CONFUSION MATRICES")
print("=" * 70)

for strategy in ['zero_shot', 'chain_of_thought', 'few_shot']:
    row = summary_df[summary_df['Strategy'] == strategy].iloc[0]
    print(f"\n{strategy}:")
    print("                    LLM Prediction")
    print("                    Not Consist.  Consistent")
    print(f"Human  Not Consist.     {row['TN']:^6}      {row['FP']:^6}")
    print(f"       Consistent       {row['FN']:^6}      {row['TP']:^6}")

In [ ]:
# Visualization - ALL STRATEGIES
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Cohen's Kappa comparison bar chart
strategies = summary_df['Strategy'].values
kappas = summary_df['Kappa'].values
colors = ['gold' if k == max(kappas) else 'steelblue' for k in kappas]

axes[0, 0].bar(strategies, kappas, color=colors)
axes[0, 0].set_ylabel("Cohen's Kappa")
axes[0, 0].set_title("Cohen's Kappa by Prompt Strategy")
axes[0, 0].set_ylim(0, 1)
axes[0, 0].axhline(y=0.6, color='green', linestyle='--', alpha=0.5)
axes[0, 0].axhline(y=0.4, color='orange', linestyle='--', alpha=0.5)
for i, (s, k) in enumerate(zip(strategies, kappas)):
    axes[0, 0].text(i, k + 0.02, f'{k:.2f}', ha='center', fontsize=10)

# 2. Agreement rate comparison
agreements = summary_df['Agreement'].values * 100

axes[0, 1].bar(strategies, agreements, color=['steelblue', 'coral', 'seagreen'])
axes[0, 1].set_ylabel('Agreement Rate (%)')
axes[0, 1].set_title('Human-LLM Agreement Rate by Strategy')
axes[0, 1].set_ylim(0, 100)
for i, a in enumerate(agreements):
    axes[0, 1].text(i, a + 1, f'{a:.0f}%', ha='center', fontsize=10)

# 3. Score distribution by human label (Zero-Shot example)
consistent_scores_zs = df[df['human_binary'] == 1]['zero_shot_score'].dropna()
not_consistent_scores_zs = df[df['human_binary'] == 0]['zero_shot_score'].dropna()
consistent_scores_cot = df[df['human_binary'] == 1]['chain_of_thought_score'].dropna()
not_consistent_scores_cot = df[df['human_binary'] == 0]['chain_of_thought_score'].dropna()

positions = [1, 2, 4, 5]
data = [not_consistent_scores_zs, consistent_scores_zs, not_consistent_scores_cot, consistent_scores_cot]
bp = axes[1, 0].boxplot(data, positions=positions, widths=0.6)
axes[1, 0].set_xticks([1.5, 4.5])
axes[1, 0].set_xticklabels(['Zero-Shot', 'Chain-of-Thought'])
axes[1, 0].axhline(y=THRESHOLD, color='red', linestyle='--', label=f'Threshold ({THRESHOLD})')
axes[1, 0].set_ylabel('LLM Consistency Score')
axes[1, 0].set_title('Score Distribution by Human Label')
axes[1, 0].legend(['Threshold', 'Not Consistent', 'Consistent'], loc='lower right', fontsize=8)

# 4. Heatmap: Strategy × Metric
metrics_data = summary_df[['Kappa', 'Agreement']].values
im = axes[1, 1].imshow(metrics_data.T, cmap='RdYlGn', aspect='auto', vmin=0, vmax=1)
axes[1, 1].set_xticks(range(len(strategies)))
axes[1, 1].set_xticklabels(strategies, rotation=45, ha='right')
axes[1, 1].set_yticks([0, 1])
axes[1, 1].set_yticklabels(["Cohen's Kappa", 'Agreement'])
axes[1, 1].set_title('Strategy Performance Heatmap')

for i in range(2):
    for j in range(len(strategies)):
        val = metrics_data[j, i]
        axes[1, 1].text(j, i, f'{val:.2f}', ha='center', va='center', 
                        color='white' if val < 0.5 else 'black', fontsize=11)

plt.colorbar(im, ax=axes[1, 1])
plt.tight_layout()
plt.savefig('inter_rater_agreement.png', dpi=150)
plt.show()

print("\n✓ Saved visualization to inter_rater_agreement.png")

## Conclusion

### Key Findings:

Cohen's Kappa measures how well each LLM judge strategy aligns with human judgment when classifying author accounts as "Consistent" or "Not Consistent" with Lincoln's primary source claims.

**Cross-Study Validation:**
- This analysis validates findings from **3Ba (Prompt Robustness)** and **3Bb (Self-Consistency)**
- Zero-Shot and Few-Shot show higher agreement with human labels than Chain-of-Thought

**Factors affecting agreement:**
1. **Threshold sensitivity**: The threshold (default=70) significantly affects binary classification
2. **Sample size**: With 10 annotated pairs, results are indicative but not definitive
3. **Edge cases**: Hay & Nicolay's Volume 1 contains minimal content about later events, making it an obvious "Not Consistent" case

### Recommendation:

Choose the strategy with the **highest Cohen's Kappa** for production use, as it best aligns with human expert judgment.

## Limitations

### 1. Small Sample Size
- Only **10 annotated pairs** limits statistical power
- Cohen's Kappa is sensitive to sample size — results are indicative, not definitive
- A larger annotation effort (50-100 pairs) would provide more robust estimates

### 2. Single Human Annotator
- No inter-human agreement baseline measured
- Cannot distinguish between LLM error and human subjectivity
- Future work should include multiple annotators to establish human agreement ceiling

### 3. Threshold Sensitivity
- Binary classification uses a fixed threshold (score ≥ 70 = Consistent)
- Different thresholds would yield different Kappa values
- The threshold was chosen heuristically, not empirically optimized

### 4. Class Imbalance
- Dataset is **80% Consistent** (8/10 pairs)
- This imbalance inflates simple agreement metrics
- Cohen's Kappa corrects for this, revealing CoT's lack of discriminative power (κ = 0)

### 5. Event Coverage
- Annotations cover only 4 events (excludes Ford's Theatre Assassination)
- Some author-event combinations may be underrepresented
- Problematic comparisons (identified in 3Bb) were not specifically targeted for annotation

---

## Conclusion

### Key Findings

| Metric | Zero-Shot | CoT | Few-Shot |
|--------|-----------|-----|----------|
| **Cohen's Kappa** | **0.64** | 0.00 | **0.64** |
| **Agreement Rate** | 90% | 80% | 90% |
| **Discriminative Power** | ✓ Yes | ✗ No | ✓ Yes |

### The Chain-of-Thought Problem

Despite explicit reasoning steps, CoT showed **zero agreement beyond chance** (κ = 0):
- Predicted ALL 10 pairs as "Consistent" (all scores ≥ 70)
- 80% agreement is entirely due to class imbalance
- **Systematic positive bias** — CoT cannot identify contradictions

### Cross-Study Synthesis

| Study | Best Strategy | Worst Strategy |
|-------|---------------|----------------|
| **3Ba: Prompt Robustness** | Zero-Shot | Few-Shot (outliers) |
| **3Bb: Self-Consistency** | Few-Shot | Zero-Shot (high variance) |
| **3Bc: Inter-Rater Agreement** | Zero-Shot / Few-Shot | CoT (no discrimination) |

### Final Recommendation

**Zero-Shot prompting** provides the best overall balance:
- ✓ High human alignment (κ = 0.64)
- ✓ Stable across different prompts (3Ba)
- ✓ Discriminates between Consistent and Not Consistent cases
- ⚠ Moderate run-to-run variance (addressable with temperature=0)

For production deployment, use **Zero-Shot with temperature=0** for maximum reliability.

---

*This study is part of the Memory Machines project evaluating LLM reliability for historical document analysis.*

In [ ]:
# Save Results - ALL STRATEGIES
OUTPUT_DIR = Path("output")
OUTPUT_DIR.mkdir(exist_ok=True)

# Save inter-rater agreement results for all strategies
agreement_results = {
    "threshold": THRESHOLD,
    "total_pairs": len(df),
    "strategies": {},
    "best_strategy": best_strategy['Strategy'],
    "best_kappa": float(best_strategy['Kappa'])
}

for _, row in summary_df.iterrows():
    agreement_results["strategies"][row['Strategy']] = {
        "cohens_kappa": float(row['Kappa']),
        "simple_agreement": float(row['Agreement']),
        "confusion_matrix": {
            "TN": int(row['TN']), "FP": int(row['FP']),
            "FN": int(row['FN']), "TP": int(row['TP'])
        }
    }

# Add detailed results
agreement_results["detailed_results"] = df[[
    'id', 'event', 'author', 'human_rating', 'human_binary',
    'zero_shot_score', 'zero_shot_binary',
    'chain_of_thought_score', 'chain_of_thought_binary',
    'few_shot_score', 'few_shot_binary'
]].to_dict('records')

with open(OUTPUT_DIR / "inter_rater_agreement.json", 'w') as f:
    json.dump(agreement_results, f, indent=2)

print(f"✓ Saved results to {OUTPUT_DIR / 'inter_rater_agreement.json'}")

# Also save summary CSV
summary_df.to_csv(OUTPUT_DIR / "inter_rater_summary.csv", index=False)
print(f"✓ Saved summary to {OUTPUT_DIR / 'inter_rater_summary.csv'}")